In [11]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [12]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [13]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [14]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [15]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [16]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [17]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [18]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 112,
 'tn': 2596,
 'fp': 41,
 'fn': 251,
 'misclassification_rate': 0.09733333333333333,
 'false_positive_rate': 0.015547971179370497,
 'false_negative_rate': 0.6914600550964187}

### Check results on the test set (new data not yet seen by the model)

In [19]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 31,
 'tn': 861,
 'fp': 13,
 'fn': 95,
 'misclassification_rate': 0.108,
 'false_positive_rate': 0.014874141876430207,
 'false_negative_rate': 0.753968253968254}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

The model has 10.8% misclassification rate, which means that 89.2% of sample data was сorrectly classified. The false negative rate is 75.4%, so it may fail to identify a significant number of bots. The false positive rate is 14.9%, so it may identify a legitimate user as a bot. Based on this information I would say that the model is passably good, but not perfect.

### What are potential ramifications of false positives from the model?

False positive occurs when a legitimate user is identified as a bot, this may lead to penalty of a real user with no reason.

### What are potential ramifications of false negatives from the model?

False positive occurs when a bot is identified as a real user, which allows malicious accounts to stay on the platform and perform negative actions.